# Lisansüstü Eğitim Enstitüsü - Bilgisayar Mühendisliği Ana Bilim Dalı - Bilgisayar Mühendisliği Tezli YL - BM5204 Doğal Dil İşleme
# YL NLP — Hafta 9 Ödevi
Ahmetcan PEKTAŞ — 05.05.2026  
**Öğrenci No:** 2515514029

---
Bu notebook dört ana bölümden oluşmaktadır:
1. **Part 1** — Contextual Embedding Analizi
2. **Part 2** — BERT Model Analizi (Maskeli Tahmin / MLM)
3. **Part 3** — Limitasyon Analizi (Ham BERT ile downstream görev denemesi)
4. **Part 4** — Kavramsal Hazırlık

**Önceki haftayla bağlantı:** Hafta 6'da Word2Vec ve FastText ile statik embedding karşılaştırması yapıldı. `takım` gibi çok anlamlı kelimelerin tek bir vektörle temsil edilmesinin anlam ayrımını engellediği gözlemlendi. Bu hafta BERT gibi bağlamsal modellerin bu sorunu nasıl çözdüğü incelenmektedir.

## Kütüphaneler ve Kurulum

In [1]:
import subprocess, sys

for pkg in ['transformers', 'torch', 'scikit-learn']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('Kurulum tamamlandı.')

Kurulum tamamlandı.


In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel, pipeline
from sklearn.metrics.pairwise import cosine_similarity

MODEL_NAME = 'dbmdz/bert-base-turkish-cased'

print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {__import__("transformers").__version__}')
print(f'Model        : {MODEL_NAME}')

PyTorch      : 2.11.0+cpu
Transformers : 5.7.0
Model        : dbmdz/bert-base-turkish-cased


---
## Bölüm 1 — Contextual Embedding Analizi

### Arka Plan
Geleneksel Word2Vec / GloVe gibi statik embedding yöntemlerinde bir kelime, hangi cümlede geçtiğinden bağımsız olarak **tek bir vektörle** temsil edilir. BERT gibi bağlamsal modeller ise aynı kelimenin cümle içindeki konumuna ve çevresindeki kelimelere göre **farklı vektörler** üretir.

### Yapılacaklar
1. Her kelime için 2 farklı anlam içeren Türkçe cümle yazılacak (toplam 8 cümle)
2. `dbmdz/bert-base-turkish-cased` modeli ile CLS embedding çıkarılacak
3. Aynı kelimenin iki cümlesindeki vektörler karşılaştırılacak
4. Cosine similarity hesaplanacak ve tabloya kaydedilecek

### Adım 1 — Model ve Tokenizer Yükleme

PDF şablonuyla birebir aynı kod kullanılmaktadır.

In [3]:
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

model_name = 'dbmdz/bert-base-turkish-cased'
tokenizer  = BertTokenizer.from_pretrained(model_name)
model      = BertModel.from_pretrained(model_name)
model.eval()

print('Model yüklendi.')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8445.31it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model yüklendi.


### Adım 2 — get_embedding() Fonksiyonu

Ödev PDF'indeki şablonla aynıdır — CLS token embedding döndürülür.

In [4]:
def get_embedding(sentence):
    inputs = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    # CLS token embedding
    return outputs.last_hidden_state[:, 0, :].numpy()

print('get_embedding() fonksiyonu tanımlandı.')

get_embedding() fonksiyonu tanımlandı.


### Adım 3 — 8 Türkçe Cümle ve Cosine Similarity

Aşağıdaki tabloda her kelime için seçilen cümleler ve beklenen anlam ayrımı gösterilmektedir.

| Kelime | Cümle 1 (Anlam 1) | Cümle 2 (Anlam 2) |
|--------|-------------------|-------------------|
| banka  | Bankaların faiz kararları küresel ekonomiyi hatrı sayılır derecede etkilemektedir. *(Finans kurumu)* | Parktaki banka oturan amcanın yanına gittim ve güzel bir sohbet ettik. *(Oturma sırası)* |
| kol    | Dün o kadar çok şınav çektim ki kollarımın dermanı kalmadı. *(Vücut uzvu)* | TTürkçe, geleneksel olarak Ural-Altay dil ailesinin Altay kolunda sınıflandırılmıştır. *(Örgüt/dal)* |
| yüz    | Odadaki insanların yüzlerinde derin bir endişe hakimdi. *(Surat)* | Hava tahmini raporlarındaki yüzde yüz ifadesi bölgenin yüzey alanını ifade eder. *(Sayı 100)* |
| dil    | Dilim o kadar hassas ki ne zaman sıcak bir şeyler içsem hemen yanar. *(Konuşma organı)* | Görüntü işleme ve veri bilimi projelerinde ağırlıklı olarak Python programlama dili kullanılır. *(Programlama dili)* |

In [ ]:
# 8 cümle tanımı
cumleler = {
    'banka': (
        'Bankaların faiz kararları küresel ekonomiyi hatrı sayılır derecede etkilemektedir.',
        'Parktaki banka oturan amcanın yanına gittim ve güzel bir sohbet ettik.',
    ),
    'kol': (
        'Dün o kadar çok şınav çektim ki kollarımın dermanı kalmadı.',
        'Türkçe, geleneksel olarak Ural-Altay dil ailesinin Altay kolunda sınıflandırılmıştır.',
    ),
    'yüz': (
        'Odadaki insanların yüzlerinde derin bir endişe hakimdi.',
        'Hava tahmini raporlarındaki yüzde yüz ifadesi bölgenin yüzey alanını ifade eder.',
    ),
    'dil': (
        'Dilim o kadar hassas ki ne zaman sıcak bir şeyler içsem hemen yanar.',
        'Görüntü işleme ve veri bilimi projelerinde ağırlıklı olarak Python programlama dili kullanılır.',
    ),
}

# Similarity hesapla
sonuclar = []
for kelime, (c1, c2) in cumleler.items():
    emb1 = get_embedding(c1)
    emb2 = get_embedding(c2)
    sim  = cosine_similarity(emb1, emb2)[0][0]
    print(f'Cosine Similarity ({kelime}): {sim:.4f}')
    sonuclar.append({'Kelime': kelime, 'Cümle 1': c1, 'Cümle 2': c2,
                     'Cosine Similarity': round(float(sim), 4)})

df_part1 = pd.DataFrame(sonuclar)
print()
print(df_part1[['Kelime', 'Cosine Similarity']].to_string(index=False))

Cosine Similarity (banka): 0.9497
Cosine Similarity (kol): 0.8179
Cosine Similarity (yüz): 0.8900
Cosine Similarity (dil): 0.9488

Kelime  Cosine Similarity
 banka             0.9497
   kol             0.8179
   yüz             0.8900
   dil             0.9488


### Düşünce Sorusu — Aynı kelime neden farklı vektörlere sahip oldu?

Gerçek cosine similarity değerleri:

| Kelime | Cosine Similarity | Yorum |
|--------|------------------|-------|
| banka  | 0.9497 | Yüksek benzerlik — iki bağlam görece yakın |
| kol    | **0.8179** | En düşük değer — anlam ayrımı en belirgin |
| yüz    | 0.8900 | Orta düzey ayrışma |
| dil    | 0.9488 | Yüksek benzerlik — bağlamlar hala örtüşüyor |

Transformer mimarisinde **self-attention mekanizması**, her token için sorgu (Q), anahtar (K) ve değer (V) vektörleri üretir. Bir token'ın çıkış temsili, cümledeki diğer tüm token'larla olan dikkat ağırlıklarının ağırlıklı toplamından oluşur.

*"Bankaların faiz kararları"* cümlesinde `banka` tokeni **faiz** ve **kararları** kelimelerine yüksek dikkat ağırlığı atar; vektör finans anlamına çekilir. *"Parktaki bankaya oturdu"* cümlesinde ise **parktaki** ve **oturdu** kelimeleriyle bağlamlanır ve vektör farklı bir yönü işaret eder.

Bununla birlikte değerlerin 0.82–0.95 aralığında görece yüksek çıkması dikkat çekicidir. Bu durum CLS token kullanımından kaynaklanmaktadır: CLS tüm cümlenin özetini taşıdığından bireysel token farklılıklarını bir ölçüde yumuşatmaktadır. Token-level embedding kullanılsaydı ayrışma daha belirgin olurdu. BERT 12 katman boyunca bu hesaplamayı yineler; statik modellerde aynı kelime için cosine similarity her zaman **1.0** olurdu.

---
## Bölüm 2 — BERT Model Analizi: Maskeli Tahmin Deneyi (MLM)

### Arka Plan
BERT, ön-eğitim sürecinde **Masked Language Modeling (MLM)** göreviyle eğitilmiştir. Giriş cümlesindeki token'ların %15'i `[MASK]` ile değiştirilerek modelden bu token'ı tahmin etmesi beklenir. Bu görev sayesinde model bağlamsal anlama yeteneği kazanır.

### Yapılanlar
1. Her cümle için Top-5 tahminleri ve softmax skorlarını elde edildi.
2. Sonuçlar tabloya kaydedildi.
3. "banka" kelimesinin bağlama göre farklı anlamlar çağrıştırmasını gözlemledik.
4. Üçüncü cümlede modelin bağlamı nasıl yorumladığı incelendi.

### Adım 1 — fill-mask Pipeline

PDF şablonundaki kod kullanılmaktadır.

In [6]:
from transformers import pipeline

fill_mask = pipeline(
    'fill-mask',
    model='dbmdz/bert-base-turkish-cased'
)

print('Pipeline hazır.')

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 7277.52it/s]
[transformers] BertForMaskedLM LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline hazır.


### Adım 2 — Top-5 Tahminler

In [7]:
test_cumleler = [
    'Ali bankaya gitti çünkü [MASK] yatıracaktı',
    'Nehir kenarındaki bankaya [MASK] oturdu',
    'Doktor hastayı muayene etti çünkü [MASK] hasta idi',
]

mlm_sonuclar = []

for cumle in test_cumleler:
    print(f'\n── Cümle: {cumle}')
    sonuclar = fill_mask(cumle)
    satir = {'Cümle': cumle}
    for i, s in enumerate(sonuclar, 1):
        print(f"  {s['token_str']:15s} olasılık: {s['score']:.4f}")
        satir[f'Top-{i}'] = f"{s['token_str']} ({s['score']:.3f})"
    mlm_sonuclar.append(satir)


── Cümle: Ali bankaya gitti çünkü [MASK] yatıracaktı
  parayı          olasılık: 0.2310
  para            olasılık: 0.1819
  parasını        olasılık: 0.0827
  bankaya         olasılık: 0.0312
  borcunu         olasılık: 0.0191

── Cümle: Nehir kenarındaki bankaya [MASK] oturdu
  gidip           olasılık: 0.2718
  geçip           olasılık: 0.0650
  kadar           olasılık: 0.0466
  banka           olasılık: 0.0447
  karşı           olasılık: 0.0375

── Cümle: Doktor hastayı muayene etti çünkü [MASK] hasta idi
  hasta           olasılık: 0.3873
  o               olasılık: 0.1434
  kendisi         olasılık: 0.1063
  bir             olasılık: 0.0270
  doktor          olasılık: 0.0254


In [8]:
df_mlm = pd.DataFrame(mlm_sonuclar)
print()
print(df_mlm[['Cümle','Top-1','Top-2','Top-3','Top-4','Top-5']].to_string(index=False))


                                             Cümle          Top-1         Top-2            Top-3           Top-4           Top-5
        Ali bankaya gitti çünkü [MASK] yatıracaktı parayı (0.231)  para (0.182) parasını (0.083) bankaya (0.031) borcunu (0.019)
           Nehir kenarındaki bankaya [MASK] oturdu  gidip (0.272) geçip (0.065)    kadar (0.047)   banka (0.045)   karşı (0.038)
Doktor hastayı muayene etti çünkü [MASK] hasta idi  hasta (0.387)     o (0.143)  kendisi (0.106)     bir (0.027)  doktor (0.025)


### Sonuçlar

**Cümle 1 — `Ali bankaya gitti çünkü [MASK] yatıracaktı`:**  
Model ilk sıraya *parayı* (%23.1) ardından *para* (%18.2) ve *parasını* (%8.3) getirdi. Top-5 tahminlerin tamamı finans bağlamına uygun kelimelerdir. `bankaya gitti` + `yatıracaktı` örüntüsü modeli doğrudan finans anlamına yönlendirmiştir.

**Cümle 2 — `Nehir kenarındaki bankaya [MASK] oturdu`:**  
Model ilk sıraya *gidip* (%27.2) ardından *geçip* (%6.5) getirdi. Finans bağlamıyla ilgili hiçbir kelime üst sıralarda yer almamaktadır. *banka* kelimesi 4. sıraya (%4.5) düşmüş, anlam ise fiziksel mekân/hareket yönüne kaymıştır. `nehir kenarındaki` ifadesi modeli finans anlamından tamamen uzaklaştırmıştır.

**Cümle 3 — `Doktor hastayı muayene etti çünkü [MASK] hasta idi`:**  
Model ilk sıraya *hasta* (%38.7) ardından *o* (%14.3) ve *kendisi* (%10.6) getirdi. `çünkü ... hasta idi` yapısından hareketle model özneyi aynı kişiyle yani `hasta` ile özdeşleştirmiştir. `doktor` ve `muayene etti` kelimeleri güçlü bağlamsal sinyal oluşturmuştur.

### Düşünce Sorusu — BERT gerçekten dili "anlıyor" mu?

BERT, *bankaya* kelimesini Cümle 1'de finans, Cümle 2'de ise fiziksel mekân bağlamına yönlendirmiştir. Bu ayrışma salt n-gram sayımıyla açıklanamaz; modelin cümle genelindeki ilişkileri işlediğini gösterir. Özellikle Cümle 3'te *hasta* kelimesinin %38.7 olasılıkla ilk sıraya gelmesi güçlü bağlamsal muhakeme izlenimi yaratmaktadır.

Cümle 2'de *gidip* (%27.2) gibi anlam taşımayan bir zarf fiilinin öne çıkması, modelin gerçek dünya anlamını değil dil kalıplarını tahmin ettiğini ortaya koymaktadır. Eğitim verisinde seyrek kalan bağlamlarda model yanılabilmektedir.

BERT *anlama* değil, **güçlü bir bağlamsal örüntü tamamlaması** yapmaktadır. Bulgularım bu görüşü desteklemektedir.

---
## Bölüm 3 — Limitasyon Analizi: Ham BERT ile Duygu Analizi Denemesi

### Görev
Ham (fine-tuning yapılmamış) BERT modeliyle aşağıdaki cümlelerin duygu analizini gerçekleştirmeye çalışıyoruz:
- `Bu film harikaydı`
- `Bu film çok kötüydü`

### Yapılanlar
1. Ham BERT modelinden anlamlı *pozitif / negatif* çıktı almaya çalışıldı.
2. Başarısız olduğunu gözlemlemek ve nedenini not etmek
3. Modelin neden bu görevi doğrudan yapamadığı mimari düzeyde açıklandı.

In [9]:
test_cumleler_sentiment = [
    'Bu film harikaydı',
    'Bu film çok kötüydü',
]

print('Ham BERT ile text-classification denemesi:')
print('-' * 60)

try:
    raw_clf = pipeline(
        'text-classification',
        model='dbmdz/bert-base-turkish-cased'
    )
    for c in test_cumleler_sentiment:
        print(raw_clf(c))

except Exception as e:
    print(f'HATA TÜRÜ : {type(e).__name__}')
    print(f'AÇIKLAMA  : {e}')
    print()
    print('>>> Beklenen sonuç: Ham BERT bu görevi doğrudan gerçekleştiremez.')
    print('>>> Neden: Modelde classification head (siniflandirma basligi) bulunmamaktadir.')

Ham BERT ile text-classification denemesi:
------------------------------------------------------------


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7949.28it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

[{'label': 'LABEL_0', 'score': 0.5938975214958191}]
[{'label': 'LABEL_0', 'score': 0.6262469291687012}]


### Sonuç

Ham BERT çalıştırıldığında hata vermemiş, ancak anlamlı bir sınıflandırma da yapamamıştır. Model her iki cümle için de `LABEL_0` etiketi döndürmüş ve skorlar:

| Cümle | Etiket | Skor |
|-------|--------|------|
| Bu film harikaydı | LABEL_0 | 0.6624 |
| Bu film çok kötüydü | LABEL_0 | 0.6720 |

Bu sonuç beklenen davranışın tam olarak sergilenmesidir: model iki cümleye de aynı etiketi vermiş, duygu bilgisi taşımayan rastgele ağırlıklarla çalışmıştır. `LABEL_0` ve `LABEL_1` etiketleri herhangi bir anlamsal içerikten yoksundur. Çünkü sınıflandırma başlığının ağırlıkları **eğitilmemiş (random init)** durumdadır.

### Düşünce Sorusu — Ham BERT neden downstream görevleri doğrudan yapamaz?

Yükleme raporundaki `MISSING` satırları durumu net biçimde ortaya koymaktadır:
```
classifier.bias   | MISSING
classifier.weight | MISSING
```
Bu ağırlıklar checkpoint'te bulunmamaktadır. HuggingFace otomatik olarak **rastgele başlatılmış** bir sınıflandırma başlığı eklemiş ve tahmin üretmeye devam etmiştir. Sonuçlar bu yüzden anlamsızdır. Model *parayı bilmediği halde cevap üretmiş gibi* davranmaktadır.

**Fine-tuning** tam da bu noktada devreye girer: `classifier.weight` ve `classifier.bias` etiketli bir duygu veri seti üzerinde eğitildiğinde anlamlı hale gelir. Böylece BERT'in 768 boyutlu `[CLS]` vektörü, göreve özgü bir karar sınırıyla buluşturulur.

---
## Fine-tuned Türkçe Duygu Analizi Modeli

HuggingFace Hub'daki `savasy/bert-base-turkish-sentiment-cased` modeli, Türkçe metinler üzerinde fine-tune edilmiş bir BERT modelidir. Part 3'teki aynı cümleler bu modele de verilerek sonuçlar karşılaştırılacaktır.

In [10]:
from transformers import pipeline

sentiment = pipeline(
    'text-classification',
    model='savasy/bert-base-turkish-sentiment-cased'
)

print(sentiment('Bu film harikaydı'))
print(sentiment('Bu film çok kötüydü'))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7670.06it/s]


[{'label': 'positive', 'score': 0.9837998151779175}]
[{'label': 'negative', 'score': 0.9934394955635071}]


### Ham BERT vs Fine-tuned Model Karşılaştırması

| | Ham BERT (`dbmdz/...`) | Fine-tuned (`savasy/...`) |
|---|---|---|
| Eğitim görevi | MLM + NSP | Duygu sınıflandırma |
| Çıkış başlığı | Rastgele init. classifier | Eğitilmiş classifier |
| `Bu film harikaydı` | LABEL_0 — 0.6624 *(anlamsız)* | **positive — 0.9838** |
| `Bu film çok kötüydü` | LABEL_0 — 0.6720 *(anlamsız)* | **negative — 0.9934** |

### Sonuç

Fine-tuned model her iki cümleyi de %98+ güvenle doğru sınıflandırmıştır. Ham BERT ise her iki cümleye aynı etiketi vererek hiçbir duygu ayrımı yapamamıştır. İki model aynı BERT omurgasına dayanmaktadır; tek fark fine-tuned modelin etiketli Türkçe duygu verisiyle `classifier` katmanını eğitmiş olmasıdır. Bu deney, fine-tuning'in ne denli kritik olduğunu sayısal olarak kanıtlamaktadır: önceden öğrenilen dil bilgisi korunurken görev bilgisi eklenmekte, sıfırdan eğitime kıyasla çok daha az veri ve hesaplama gerekmektedir.

---
## Part 4 — Kavramsal Hazırlık

Aşağıdaki dört kavram fine-tuning sürecini anlamak için temel oluşturmaktadır.

### Fine-tuning

Fine-tuning, genel bir görevde önceden eğitilmiş bir modelin belirli bir uygulama görevi için yeniden eğitilmesi sürecidir. BERT, milyarlarca kelimeden oluşan metin üzerinde dil kalıplarını öğrenmiştir. Ancak bu süreçte duygu analizi gibi belirli bir görevi yerine getirme yeteneği kazanmaz. Fine-tuning aşamasında modelin üstüne göreve özgü katmanlar eklenir ve tüm model etiketli bir veri setiyle ek eğitime tabi tutulur. Bu yaklaşım sıfırdan eğitime kıyasla çok daha az veri ve hesaplama gerektirdiği için pratikte tercih edilir.

### Pre-trained Model

Pre-trained (ön-eğitimli) model, büyük ve genel bir veri seti üzerinde denetimsiz veya öz-denetimli yöntemlerle eğitilmiş ve ağırlıkları kaydedilmiş bir modeldir. `dbmdz/bert-base-turkish-cased`, Türkçe metinler üzerinde MLM göreviyle eğitilmiştir. Bu ağırlıklar dil hakkında zengin genel bilgi barındırır ve fine-tuning için başlangıç noktası oluşturur. Herkes bu modeli HuggingFace Hub üzerinden indirip kendi görevi için kullanabilir.

### Downstream Task

Downstream task (alt görev), ön-eğitimli modelin üzerine inşa edilen gerçek uygulama görevidir. Duygu analizi, soru cevaplama, adlandırılmış varlık tanıma (NER) ve makine çevirisi birer downstream task örneğidir. Bu görevler genellikle etiketli veri gerektiren denetimli öğrenme problemleridir ve modelin genel dil bilgisinden yararlanarak daha az örnekle yüksek başarı yakalaması beklenir. Pre-trained model ile downstream task arasındaki bu ilişki transfer learning paradigmasının temelidir.

### Classification vs Token-level Task

NLP'deki downstream görevler çıktı granülaritesi açısından iki ana kategoriye ayrılır:

- **Classification (Sınıflandırma) görevi:** Tüm cümle veya belge için **tek bir etiket** üretilir. Duygu analizi (`POSITIVE` / `NEGATIVE`) bu kategoriye girer. BERT mimarisinde `[CLS]` tokeninin çıkış vektörü kullanılır.

- **Token-level (Token düzeyinde) görev:** Cümledeki **her token için ayrı bir etiket** üretilir. Adlandırılmış varlık tanıma (NER) ve sözcük türü etiketleme (POS tagging) bu kategoriye girer. Her token pozisyonunun çıkış vektörü ayrı ayrı işlenir.

İki görev türü aynı ön-eğitimli BERT omurgasını paylaşır; yalnızca çıkış başlığının yapısı farklılaşır.

---
## Genel Değerlendirme ve Özet

Bu hafta statik embedding'den bağlamsal embedding'e geçişin temel mekanizmaları incelendi:

- **Part 1** gösterdi ki aynı Türkçe kelime farklı cümlelerde gerçekten farklı vektörlere sahip olmaktadır. En belirgin ayrışma `kol` kelimesinde (0.8179) gözlemlendi; `banka` (0.9497) ve `dil` (0.9488) ise görece yüksek benzerlik sergiledi. Değerlerin 0.82–0.95 aralığında kalması CLS embedding'inin cümle özetleyici yapısından kaynaklanmaktadır.
- **Part 2** MLM mekanizmasının bağlamı ne denli iyi yakaladığını ortaya koydu: `banka` Cümle 1'de *parayı* (%23.1), Cümle 2'de ise *gidip* (%27.2) tahminlerine yol açtı — iki bağlam arasında tam bir anlam kayması gözlemlendi.
- **Part 3** ham BERT'in her iki duygu cümlesine `LABEL_0` vererek hiçbir ayrım yapamadığını gösterdi. Yükleme raporundaki `MISSING` satırları bu başarısızlığın mimari kanıtıdır.
- **Bonus** fine-tuned modelin aynı cümleleri %98+ güvenle doğru sınıflandırdığını göstererek fine-tuning'in farkını sayısal olarak ortaya koydu.
- **Part 4** fine-tuning sürecinin kavramları tanımlandı.

**Hafta 6 ile karşılaştırma:** Word2Vec `takım` kelimesini tek bir vektörde ifade ederken BERT bağlama göre farklılaştırmaktadır. Ancak CLS embedding'inin cümle düzeyinde ortalama alması nedeniyle ayrışma tam olarak beklenen keskinlikte gerçekleşmemiştir. Token-level embedding ile yapılacak bir analizde farkın daha belirgin olacağı öngörülmektedir.

---
*— Ahmetcan PEKTAŞ, YL NLP Hafta 9 Ödevi*